# Mini-Project 3: Lake Victoria Fish Stock & Export Risk Model
**Scenario.** A fish-export co-operative in Jinja asks (a) is our harvest rate sustainable? and (b) how risky is our revenue? The earlier model used Fibonacci numbers for the stock and flagged risk when the revenue **variance** exceeded 50,000. This notebook replaces both with a logistic stock model and a coefficient-of-variation / Value-at-Risk assessment, and explains what was wrong with the old approach.

Logic lives in `src/fish_stock.py` (`FishStock`, `PriceModel`, `Fishery`, `RiskAssessor`); tests are in `tests/test_fish_stock.py`.
**Units:** stock/harvest in tonnes, price in UGX/kg, revenue in UGX; one time-step = one week.

In [ ]:
# --- Google Colab setup: does nothing when you run on your own computer ---
import os, subprocess, sys

REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"   # change this after you push the repo to GitHub

if "google.colab" in sys.modules and not os.path.isdir("src"):
    if "YOUR_USERNAME" in REPO_URL:
        raise RuntimeError("Colab cannot find the repo's src/ folder. Either edit REPO_URL above, or upload and "
                           "unzip the repo first (see the README section 'Run in Google Colab').")
    repo_dir = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", "-q", REPO_URL, repo_dir], check=True)
    os.chdir(repo_dir)
print("Working folder:", os.getcwd())

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from src.fish_stock import (
    FishStock, Fishery, PriceModel, RiskAssessor,
    fibonacci, plot_revenue_hist, plot_stock_paths, weekly_revenue,
)

SEED = 42
Path("figures").mkdir(exist_ok=True)
stock_model = FishStock(r=0.4, K=10_000, n0=4_000, h=0.10)   # baseline harvest rate h = 0.10
prices = PriceModel(start=12_000, sigma=350, low=9_000, high=16_000)
assessor = RiskAssessor()                                    # CV thresholds 0.05 / 0.15, 5% VaR
stock_model, prices

## Task 1: The Fibonacci baseline

In [ ]:
fib = fibonacci(15)
print(fib)
print("ratios of successive terms:", [round(fib[i+1]/fib[i], 3) for i in range(9, 14)], "-> golden ratio 1.618")

**Why unbounded Fibonacci growth is biologically unrealistic.** Fibonacci numbers grow by a factor that tends to 1.618 every step, forever, so the "stock" would grow by ~62% per period with no ceiling. Real fish populations are limited by food, habitat and oxygen: as the stock approaches the lake's carrying capacity, growth per fish slows (density dependence), which the logistic model captures and Fibonacci cannot. The sequence also has no harvesting or natural mortality term, so it cannot say anything about sustainability, and it is deterministic, so it carries no uncertainty. Lake Victoria's own Nile perch fishery has shown boom-and-decline patterns, not endless growth.

## Task 2: `FishStock`: logistic growth with harvesting

In [ ]:
path = stock_model.simulate(weeks=52)
print("N(0) =", path.stock[0], "| N(1) =", round(path.stock[1], 1), "(hand calc: 4000 + 0.4*4000*0.6 - 0.1*4000 = 4560)")
print("N(52) =", round(path.stock[-1], 1), "tonnes")
print("theoretical equilibrium K(1 - h/r) =", stock_model.equilibrium_stock)
print(f"weekly harvest: first {path.harvest_tonnes[0]:.0f} t, last {path.harvest_tonnes[-1]:.0f} t")

## Task 3: `PriceModel` and weekly revenue

In [ ]:
fishery = Fishery(stock_model, prices)
price_path = prices.simulate(52, seed=SEED)[0]
revenue = weekly_revenue(path.harvest_tonnes, price_path)
assert np.allclose(revenue, fishery.revenue(52, 1, SEED)[0])      # composition gives the same answer
print(f"price: start {price_path[0]:,.0f}, min {price_path.min():,.0f}, max {price_path.max():,.0f}, mean {price_path.mean():,.0f} UGX/kg")
print(f"week-1 revenue = {path.harvest_tonnes[0]:.0f} t x 1000 kg x {price_path[0]:,.0f} = UGX {revenue[0]/1e9:.2f} billion")
print(f"annual revenue (this path) = UGX {revenue.sum()/1e9:,.1f} billion")

## Task 4: Descriptive statistics and why a variance threshold of 50,000 is meaningless

In [ ]:
d = assessor.describe(revenue)
for k in ("mean", "median", "variance", "stdev", "cv"):
    print(f"{k:<9}: {d[k]:,.4g}")

print("\nThe SAME variance expressed in different units, against the old threshold of 50,000:")
import statistics
for label, scale in (("UGX^2", 1), ("UGX-million^2", 1e6), ("UGX-billion^2", 1e9)):
    v = statistics.variance(revenue / scale)
    print(f"  {label:<14}: variance = {v:>12,.4g}  -> {'FLAGGED risky' if v > 50_000 else 'not flagged'}")

**Why the raw threshold is meaningless.**
- Variance has units of the data **squared** (UGX²). A fixed number such as 50,000 has no meaning unless we know the unit: the identical revenue series is "risky" when measured in UGX or UGX-million and "safe" in UGX-billion (above). A risk flag must not flip because we changed units.
- It also ignores scale: a co-operative earning UGX 8 billion a week and one earning UGX 8 million a week face very different absolute variances for the same *relative* uncertainty.
- The **coefficient of variation** CV = st.dev / mean is dimensionless (units cancel), so it compares risk fairly across scales and units. That is what we use.
- A caution on the weekly CV: with a harvest that depends on a *growing or shrinking stock*, weekly revenue changes partly because of a predictable trend, not only price risk (shown in Task 6). That is why the formal risk rule below uses annual revenue across Monte Carlo price paths.

## Task 5: `RiskAssessor`: CV rule and Monte Carlo VaR
**Rule.** Apply to the CV of *annual* revenue across simulated price paths (stock is deterministic, so this isolates price uncertainty):
**CV < 5% → Low; 5–15% → Moderate; ≥ 15% → High.** The thresholds are a stated judgement call (assumption: a bad year 5% below average is absorbable; 15% below would likely exceed a thin export margin of about 10%). They should be set from the co-operative's real margin.

**VaR.** `expected − 5th percentile` of annual revenue: the shortfall below expectation that a year exceeds only 5% of the time.

In [ ]:
N_PATHS = 2_000
annual = assessor.annual_revenue_mc(fishery, n_paths=N_PATHS, seed=SEED)
var = assessor.value_at_risk(annual)
annual_cv = annual.std(ddof=1) / annual.mean()
print(f"{N_PATHS:,} price paths, baseline h = 0.10")
print(f"expected annual revenue : UGX {var['expected']/1e9:,.1f} bn")
print(f"annual revenue CV       : {annual_cv:.3f}  -> risk class: {assessor.classify(annual_cv)}")
print(f"5th percentile          : UGX {var['percentile']/1e9:,.1f} bn")
print(f"5% Value-at-Risk        : UGX {var['var']/1e9:,.1f} bn  ({var['var']/var['expected']:.1%} of expected)")

## Task 6: Scenario analysis over harvest rates and comparison with MSY = rK/4

In [ ]:
H_VALUES = (0.05, 0.10, 0.20, 0.30)
price_cv = price_path.std(ddof=1) / price_path.mean()
print(f"MSY = rK/4 = {stock_model.msy:,.0f} t/week, reached at h = r/2 = {stock_model.h_msy}\n")
hdr = (f"{'h':>5}{'N(52) t':>9}{'eq. stock':>10}{'eq. yield t/wk':>15}{'% of MSY':>9}"
       f"{'E[revenue] bn':>14}{'annual CV':>10}{'risk class':>11}{'VaR bn':>8}{'weekly CV':>10}  biological status")
print(hdr)
scenario = {}
for h in H_VALUES:
    st = stock_model.with_harvest(h)
    fy = Fishery(st, prices)
    sp = st.simulate(52)
    ann = assessor.annual_revenue_mc(fy, N_PATHS, SEED)
    v = assessor.value_at_risk(ann)
    cv = ann.std(ddof=1) / ann.mean()
    wk = assessor.describe(fy.revenue(52, 1, SEED)[0])
    scenario[h] = (sp, ann, v)
    print(f"{h:5.2f}{sp.stock[-1]:9,.0f}{st.equilibrium_stock:10,.0f}{st.equilibrium_yield:15,.0f}"
          f"{st.equilibrium_yield / st.msy:9.0%}{v['expected']/1e9:14,.1f}{cv:10.3f}{assessor.classify(cv):>11}"
          f"{v['var']/1e9:8.1f}{wk['cv']:10.3f}  {st.harvest_status}")
print(f"\nFor reference, CV of the price path alone = {price_cv:.3f}: weekly revenue CVs above differ from it because the stock trend moves revenue too.")

**Reading the table.**
- Yield rises with *h* up to *h = r/2 = 0.20*, where the equilibrium weekly catch equals MSY (1,000 t/week), then **falls**: at *h = 0.30* the stock settles at 2,500 t, below K/2 = 5,000 t, and the sustainable yield drops back to 750 t/week. That is overfishing: a bigger effort for a smaller catch.
- Expected revenue is highest at *h = 0.20* and about 17% lower at *h = 0.30*, even though the harvest *rate* is 50% higher.
- Annual revenue CV is almost identical (≈ 0.10–0.11) in every scenario because price risk does not depend on how much we catch. The CV rule therefore labels all four "Moderate": it measures **financial** risk. Sustainability needs the separate biological status (h versus r/2), shown in the last column.
- The weekly CVs differ from the price-only CV and from each other because of the stock trend (h = 0.05: stock doubling during the year; h = 0.30: stock falling).

## Task 7: Visualisation

In [ ]:
fig = plot_stock_paths({h: scenario[h][0] for h in H_VALUES}, K=stock_model.K, r=stock_model.r)
fig.savefig("figures/project3_stock_paths.png", dpi=150)
plt.show()

Stock settles at K(1 − h/r): 8,750, 7,500, 5,000 and 2,500 t. Only h ≤ 0.20 keeps the stock at or above K/2, the level that gives MSY.

In [ ]:
results = {f"h = {h:.2f} ({stock_model.with_harvest(h).harvest_status})": (scenario[h][1], scenario[h][2]) for h in H_VALUES}
fig = plot_revenue_hist(results)
fig.savefig("figures/project3_revenue_hist.png", dpi=150)
plt.show()

Each histogram shows 2,000 simulated years; the red line is the 5th percentile and the label gives the VaR. The *spread* is relatively similar across scenarios (same price risk), but the *location* moves with the harvest rate.

## Extension: closed season (no harvest for 8 weeks per year, over 5 years)
Weeks 20–27 of each year are closed (illustrative spawning window). We compare 5-year expected revenue (2,000 price paths, same seeds) with and without the closure.

In [ ]:
CLOSED = range(20, 28)
print(f"{'h':>5}{'open bn':>10}{'closed bn':>11}{'change':>8}{'end stock open':>16}{'end stock closed':>18}")
for h in H_VALUES:
    st = stock_model.with_harvest(h); fy = Fishery(st, prices)
    rev_open = fy.revenue(260, N_PATHS, SEED).sum(axis=1).mean()
    rev_closed = fy.revenue(260, N_PATHS, SEED, closed_weeks=CLOSED).sum(axis=1).mean()
    s_open = st.simulate(260).stock[-1]; s_closed = st.simulate(260, closed_weeks=CLOSED).stock[-1]
    print(f"{h:5.2f}{rev_open/1e9:10,.0f}{rev_closed/1e9:11,.0f}{rev_closed/rev_open - 1:8.1%}{s_open:16,.0f}{s_closed:18,.0f}")

**Interpretation.** For *h* ≤ 0.20 the closure simply removes 8 of 52 fishing weeks and the stock is already at or above the MSY level, so revenue falls (≈ 9–15%). For the overfished case *h* = 0.30 the closure lets the stock rebuild each year and **raises** 5-year revenue (≈ +7%). A closed season pays for itself only where fishing pressure exceeds the sustainable level; it is no substitute for choosing *h* near r/2.

## Findings & Limitations
**Findings.** Harvesting 20% of the stock per week (h = r/2) attains MSY (1,000 t/week) and the highest expected annual revenue (≈ UGX 610 bn). At h = 0.30 the catch *rate* is higher but the stock settles at 2,500 t, below K/2, so the sustainable yield falls to 750 t/week and revenue is about 17% lower: overfishing is both ecologically and financially worse. Light harvesting (h = 0.05) is safe but forgoes roughly 57% of the MSY revenue. Price risk alone gives an annual-revenue CV near 0.10–0.11 in every scenario, so all four are "Moderate" and the 5% VaR is about 15–17% of expected revenue (≈ UGX 74 bn at h = 0.10). Because price is independent of harvest, a CV rule cannot see a stock heading for collapse; sustainability must be judged separately by comparing h with r/2. The old variance threshold of 50,000 is meaningless: the same series has variance 1.4×10¹⁸ UGX², 1.4×10⁶ million², or 1.4 bn², so the verdict depends on units. A closed season cost 9–15% of revenue when h ≤ 0.20 but added about 7% when h = 0.30.

**Limitations.** r = 0.4 per week and the revenue scale are illustrative (real stocks rebuild more slowly and one co-operative would not land 1,000 t/week). Harvest is a fixed proportion, not a quota or effort model; the stock has no environmental noise; the price walk is independent of supply and clipped at its bounds, which likely understates tail risk; and revenue is not profit because costs are omitted.